In [ ]:
import os, time, math
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
# TF ma zajmować pamięć GPU stopniowo - inaczej PyTorch nie dostanie pamięci
for _gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(_gpu, True)

import torch
import keras

# Tryb szybki: True = mniej danych i epok (praca bez GPU)
SZYBKI_TRYB = os.environ.get("DL_FAST", "0") == "1"
# SZYBKI_TRYB = True   # <- odkomentuj, jeśli pracujesz bez GPU

# Ziarna losowości - każda biblioteka ma własny generator
SEED = 42
keras.utils.set_random_seed(SEED)   # Python random + NumPy + TF
np.random.seed(SEED)
torch.manual_seed(SEED)

# Urządzenie dla PyTorch: GPU jeśli jest, w przeciwnym razie CPU
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("SZYBKI_TRYB =", SZYBKI_TRYB)
print("NumPy      :", np.__version__)
print("TensorFlow :", tf.__version__, "| Keras:", keras.__version__)
print("PyTorch    :", torch.__version__)
print("GPU widziane przez TensorFlow:", tf.config.list_physical_devices("GPU"))
print("DEVICE =", DEVICE)

SZYBKI_TRYB = False
NumPy      : 2.1.3
TensorFlow : 2.21.0 | Keras: 3.13.2
PyTorch    : 2.11.0+cu130
GPU widziane przez TensorFlow: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
DEVICE = cuda


# Zadanie 1.1


In [ ]:
rng = np.random.default_rng(0)
batch_np = rng.normal(loc=[0.5, 0.2, -0.3], scale=[2.0, 0.5, 1.0],
                      size=(16, 32, 32, 3)).astype(np.float32)

# --- TensorFlow (format NHWC) ---
batch_tf = tf.constant(batch_np)
mu_tf    = tf.reduce_mean(batch_tf, axis=[0, 1, 2], keepdims=True)        # (1, 1, 1, 3)
sigma_tf = tf.math.reduce_std(batch_tf, axis=[0, 1, 2], keepdims=True)    # (1, 1, 1, 3)
z_tf = (batch_tf - mu_tf) / sigma_tf                                      # broadcasting -> (16, 32, 32, 3)

# --- PyTorch (format NCHW) ---
batch_pt = torch.from_numpy(batch_np).permute(0, 3, 1, 2)                 # (16, 3, 32, 32)
mu_pt    = batch_pt.mean(dim=(0, 2, 3), keepdim=True)                     # (1, 3, 1, 1)
sigma_pt = batch_pt.std(dim=(0, 2, 3), correction=0, keepdim=True)        # (1, 3, 1, 1)
z_pt = (batch_pt - mu_pt) / sigma_pt                                      # broadcasting -> (16, 3, 32, 32)

print("kształt statystyk TF     :", tuple(mu_tf.shape))
print("kształt statystyk PyTorch:", tuple(mu_pt.shape))
print("kształt z_tf:", tuple(z_tf.shape), "| kształt z_pt:", tuple(z_pt.shape))

kształt statystyk TF     : (1, 1, 1, 3)
kształt statystyk PyTorch: (1, 3, 1, 1)
kształt z_tf: (16, 32, 32, 3) | kształt z_pt: (16, 3, 32, 32)


### Weryfikacja

In [ ]:
# Komórka sprawdzająca
if z_tf is None or z_pt is None:
    print("Uzupełnij z_tf i z_pt.")
else:
    print("TF  średnie per kanał:", np.round(tf.reduce_mean(z_tf, axis=[0, 1, 2]).numpy(), 4))
    print("TF  std per kanał    :", np.round(tf.math.reduce_std(z_tf, axis=[0, 1, 2]).numpy(), 4))
    print("PT  średnie per kanał:", np.round(z_pt.mean(dim=(0, 2, 3)).numpy(), 4))
    z_pt_nhwc = z_pt.permute(0, 2, 3, 1).numpy()      # powrót do NHWC
    print("Zgodność TF vs PyTorch:", np.allclose(z_tf.numpy(), z_pt_nhwc, atol=1e-5))

# Zadanie 1.2

In [ ]:
def pairwise_sq_dist_np(X, Y):
    # ||x_i||^2 jako kolumna (n, 1) + ||y_j||^2 jako wiersz (1, m) - 2 X Y^T
    sq_x = (X ** 2).sum(axis=1)[:, None]        # (n, 1)
    sq_y = (Y ** 2).sum(axis=1)[None, :]        # (1, m)
    D = sq_x + sq_y - 2 * (X @ Y.T)             # broadcasting -> (n, m)
    return np.maximum(D, 0)                     # przycięcie drobnych ujemnych wartości (błędy zaokrągleń)


def pairwise_sq_dist_torch(X, Y):
    # to samo w PyTorch
    sq_x = (X ** 2).sum(dim=1).unsqueeze(1)     # (n, 1)
    sq_y = (Y ** 2).sum(dim=1).unsqueeze(0)     # (1, m)
    D = sq_x + sq_y - 2 * (X @ Y.T)
    return torch.clamp(D, min=0)

### Weryfikacja

In [ ]:
# Komórka sprawdzająca
def pairwise_sq_dist_loop(X, Y):
    D = np.zeros((len(X), len(Y)))
    for i in range(len(X)):
        for j in range(len(Y)):
            D[i, j] = np.sum((X[i] - Y[j]) ** 2)
    return D

Xs, Ys = np.random.rand(50, 8), np.random.rand(40, 8)
D_ref = pairwise_sq_dist_loop(Xs, Ys)
D_np = pairwise_sq_dist_np(Xs, Ys)
D_pt = pairwise_sq_dist_torch(torch.from_numpy(Xs), torch.from_numpy(Ys))
if D_np is not None and D_pt is not None:
    print("NumPy - zgodność z wersją pętlową:", np.allclose(D_ref, D_np))
    print("Torch - zgodność z wersją pętlową:", np.allclose(D_ref, D_pt.numpy()))

    # Pomiar czasu: pętla vs wektoryzacja
    n_big = 150 if SZYBKI_TRYB else 400
    Xb, Yb = np.random.rand(n_big, 32), np.random.rand(n_big, 32)
    t0 = time.perf_counter(); pairwise_sq_dist_loop(Xb, Yb); t_loop = time.perf_counter() - t0
    t0 = time.perf_counter(); pairwise_sq_dist_np(Xb, Yb); t_vec = time.perf_counter() - t0
    print(f"n = {n_big}: pętla {t_loop:.3f} s, wektorowo {t_vec:.5f} s, przyspieszenie ~{t_loop / max(t_vec, 1e-9):.0f}x")

# Zadanie 2.1 - Gradient funkcji dwóch zmiennych w TF i PyTorch

> Dana jest funkcja
> $$f(x, y) = \sin(x)\, y^2 + e^{x y}.$$
> 1. Wyprowadź na kartce pochodne cząstkowe $\partial f/\partial x$ i $\partial f/\partial y$ i wpisz je jako funkcje `df_dx_analit`, `df_dy_analit`.
> 2. Policz gradient w punkcie $(x, y) = (0.5, -1.0)$ za pomocą **`tf.GradientTape`** (zapisz w `grad_tf` jako listę/tablicę `[df/dx, df/dy]`).
> 3. To samo w **PyTorch** (`requires_grad=True`, `backward()`), wynik w `grad_pt`.
>
> **Kryteria sukcesu:** trzy wyniki zgodne do ok. 1e-6.
>
> **Wskazówka:** `tape.gradient(f, [x, y])` zwraca listę gradientów dla listy zmiennych.

In [ ]:
x0, y0 = 0.5, -1.0

# 1) Pochodne analityczne  (f = sin(x) * y^2 + exp(x*y)):
#    df/dx = cos(x) * y^2 + y * exp(x*y)
#    df/dy = 2 * sin(x) * y + x * exp(x*y)
df_dx_analit = lambda x, y: np.cos(x) * y ** 2 + y * np.exp(x * y)
df_dy_analit = lambda x, y: 2 * np.sin(x) * y + x * np.exp(x * y)

# 2) TensorFlow: GradientTape (float64, żeby porównanie było dokładne)
x_tf = tf.Variable(x0, dtype=tf.float64)
y_tf = tf.Variable(y0, dtype=tf.float64)
with tf.GradientTape() as tape:
    f_tf = tf.sin(x_tf) * y_tf ** 2 + tf.exp(x_tf * y_tf)
g_x, g_y = tape.gradient(f_tf, [x_tf, y_tf])
grad_tf = [g_x.numpy(), g_y.numpy()]

# 3) PyTorch: requires_grad + backward()
x_pt = torch.tensor(x0, dtype=torch.float64, requires_grad=True)
y_pt = torch.tensor(y0, dtype=torch.float64, requires_grad=True)
f_pt = torch.sin(x_pt) * y_pt ** 2 + torch.exp(x_pt * y_pt)
f_pt.backward()
grad_pt = [x_pt.grad.item(), y_pt.grad.item()]

### Weryfikacja

In [ ]:
if grad_tf is not None and grad_pt is not None and df_dx_analit is not None:
    print("analitycznie:", df_dx_analit(x0, y0), df_dy_analit(x0, y0))
    print("TF          :", grad_tf)
    print("PyTorch     :", grad_pt)

    # dodatkowo: największa różnica względem wyniku analitycznego
    ref = np.array([df_dx_analit(x0, y0), df_dy_analit(x0, y0)])
    print("maks. różnica TF      vs analitycznie:", np.max(np.abs(np.array(grad_tf) - ref)))
    print("maks. różnica PyTorch vs analitycznie:", np.max(np.abs(np.array(grad_pt) - ref)))

# Zadanie 2.2 - Sprawdzanie gradientu różnicami skończonymi (gradient check)

In [ ]:
def grad_numeryczny(f, w, h=1e-5):
    # różnica centralna dla każdej współrzędnej w
    w = np.asarray(w, dtype=np.float64)
    g = np.zeros_like(w, dtype=np.float64)
    for i in range(w.size):
        e = np.zeros_like(w, dtype=np.float64)
        e[i] = 1.0                                         # wektor e_i
        g[i] = (f(w + h * e) - f(w - h * e)) / (2 * h)
    return g

### Weryfikacja

In [ ]:
rng = np.random.default_rng(1)
A_mat = rng.normal(size=(6, 4))
w0 = rng.normal(size=4)

def f_np(w):                                   # wersja NumPy (do różnic skończonych)
    z = A_mat @ w
    return np.log(np.sum(np.exp(z))) + 0.5 * np.sum(w ** 2)

# Gradient z autograd (PyTorch, float64)
w_t = torch.tensor(w0, dtype=torch.float64, requires_grad=True)
f_t = torch.logsumexp(torch.tensor(A_mat) @ w_t, dim=0) + 0.5 * (w_t ** 2).sum()
f_t.backward()
g_auto = w_t.grad.numpy()

def blad_wzgledny(a, b):
    return np.linalg.norm(a - b) / (np.linalg.norm(a) + np.linalg.norm(b) + 1e-300)

g_num = grad_numeryczny(f_np, w0, h=1e-5)
print("autograd :", np.round(g_auto, 6))
print("numeryczny:", np.round(g_num, 6))
print("błąd względny (h=1e-5):", blad_wzgledny(g_num, g_auto))

# Błąd w funkcji kroku h
hs = np.logspace(-14, 0, 29)
bledy = [blad_wzgledny(grad_numeryczny(f_np, w0, h), g_auto) for h in hs]
plt.figure(figsize=(7, 4))
plt.loglog(hs, np.maximum(bledy, 1e-17), "o-")
plt.title("Gradient check: błąd względny vs krok h")
plt.xlabel("krok h"); plt.ylabel("błąd względny"); plt.grid(alpha=0.3, which="both"); plt.show()

# 3. Regresja liniowa od zera

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

dom = fetch_california_housing()
X_all, y_all = dom.data.astype(np.float64), dom.target.astype(np.float64)
if SZYBKI_TRYB:                       # tryb szybki: tylko 2000 pierwszych próbek
    X_all, y_all = X_all[:2000], y_all[:2000]
print("cechy:", list(dom.feature_names))
print("X:", X_all.shape, " y:", y_all.shape)

X_tr, X_te, y_tr, y_te = train_test_split(X_all, y_all, test_size=0.2, random_state=SEED)

# Standaryzacja - statystyki TYLKO ze zbioru treningowego
mu, sigma = X_tr.mean(axis=0), X_tr.std(axis=0)
X_tr_s = (X_tr - mu) / sigma
X_te_s = (X_te - mu) / sigma
n_tr, d = X_tr_s.shape
print("train:", X_tr_s.shape, " test:", X_te_s.shape)

def mse(y_true, y_pred):
    return float(np.mean((y_true - y_pred) ** 2))

# Baseline: stała = średnia z treningu
print("Baseline (średnia) MSE test:", round(mse(y_te, np.full_like(y_te, y_tr.mean())), 4))

# Rozwiązanie analityczne (najmniejsze kwadraty) - dokładne minimum na train
Xb_tr = np.hstack([X_tr_s, np.ones((n_tr, 1))])          # kolumna jedynek = bias
theta_ls = np.linalg.lstsq(Xb_tr, y_tr, rcond=None)[0]
w_ls, b_ls = theta_ls[:-1], theta_ls[-1]
loss_opt = mse(y_tr, X_tr_s @ w_ls + b_ls)
print("Najmniejsze kwadraty: MSE train =", round(loss_opt, 4),
      " MSE test =", round(mse(y_te, X_te_s @ w_ls + b_ls), 4))

## Zadanie 3.1 - Wektorowy spadek gradientu w NumPy (wiele cech)

In [ ]:
EPOKI_GD = 50 if SZYBKI_TRYB else 500

def gd_linreg(X, y, lr=0.2, epoki=EPOKI_GD):
    n, d = X.shape
    w = np.zeros(d)
    b = 0.0
    historia = []
    for ep in range(epoki):
        y_hat = X @ w + b                        # predykcja, (n,)
        err = y_hat - y                          # błąd, (n,)
        historia.append(np.mean(err ** 2))       # MSE bieżących parametrów
        dw = (2 / n) * (X.T @ err)               # gradient po w, (d,)
        db = 2 * np.mean(err)                    # gradient po b (skalar)
        w = w - lr * dw                          # krok spadku gradientu
        b = b - lr * db
    return w, b, historia

### Weryfikacja

In [ ]:
t0 = time.perf_counter()
w_gd, b_gd, hist_gd = gd_linreg(X_tr_s, y_tr, lr=0.2, epoki=EPOKI_GD)
print(f"czas: {time.perf_counter() - t0:.2f} s")
if len(hist_gd) > 0:
    print(f"MSE train GD = {hist_gd[-1]:.4f}  vs  optimum = {loss_opt:.4f}  (różnica {100 * (hist_gd[-1] / loss_opt - 1):.2f}%)")
    print("MSE test GD  =", round(mse(y_te, X_te_s @ w_gd + b_gd), 4))
    print("max |w_gd - w_ls| =", np.abs(w_gd - w_ls).max().round(4))

    plt.figure(figsize=(7, 4))
    plt.plot(hist_gd, label="GD (NumPy)")
    plt.axhline(loss_opt, color="k", ls="--", label="optimum (lstsq)")
    plt.yscale("log"); plt.title("California Housing: krzywa uczenia GD")
    plt.xlabel("epoka"); plt.ylabel("MSE train (log)"); plt.legend(); plt.grid(alpha=0.3); plt.show()

    # Które cechy mają największy wpływ? (cechy są standaryzowane, więc wagi są porównywalne)
    plt.figure(figsize=(7, 3.5))
    plt.barh(dom.feature_names, w_gd)
    plt.title("Wagi regresji liniowej (cechy standaryzowane)"); plt.xlabel("waga"); plt.ylabel("cecha")
    plt.grid(alpha=0.3); plt.show()

## Zadanie 3.2 - Regresja liniowa w TensorFlow z `GradientTape`

In [ ]:
Xtr_tf = tf.constant(X_tr_s, dtype=tf.float32)
ytr_tf = tf.constant(y_tr, dtype=tf.float32)

# 1) parametry jako tf.Variable (float32)
w_tf = tf.Variable(tf.zeros(d, dtype=tf.float32))      # (d,)
b_tf = tf.Variable(0.0, dtype=tf.float32)              # skalar
lr_tf = 0.2

# 2) + 3) pętla treningowa z GradientTape
hist_tf = []
for ep in range(EPOKI_GD):
    with tf.GradientTape() as tape:                    # tylko to, co w bloku, jest nagrywane
        y_hat = tf.linalg.matvec(Xtr_tf, w_tf) + b_tf
        loss = tf.reduce_mean((y_hat - ytr_tf) ** 2)   # MSE
    grad_w, grad_b = tape.gradient(loss, [w_tf, b_tf])
    w_tf.assign_sub(lr_tf * grad_w)                    # krok spadku gradientu
    b_tf.assign_sub(lr_tf * grad_b)
    hist_tf.append(float(loss))

### Weryfikacja

In [ ]:
if len(hist_tf) > 0:
    print(f"MSE train: NumPy {hist_gd[-1]:.5f} | PyTorch {hist_pt[-1]:.5f} | TF {hist_tf[-1]:.5f}")
    plt.figure(figsize=(7, 4))
    plt.plot(hist_gd, lw=4, alpha=0.4, label="NumPy (ręczny gradient)")
    plt.plot(hist_pt, "--", label="PyTorch autograd")
    plt.plot(hist_tf, ":", lw=2, label="TF GradientTape")
    plt.yscale("log"); plt.title("Ten sam algorytm w trzech bibliotekach")
    plt.xlabel("epoka"); plt.ylabel("MSE train (log)"); plt.legend(); plt.grid(alpha=0.3); plt.show()

# 4. Hiperparametr learning rate i liczba epok

In [ ]:
# 1) hesjan MSE, wartości własne, próg rozbieżności
Xb = np.hstack([X_tr_s, np.ones((n_tr, 1))])       # dane z kolumną jedynek (bias)
H = (2 / n_tr) * (Xb.T @ Xb)                        # hesjan: (d+1) x (d+1), symetryczny
wartosci_wlasne = np.linalg.eigvalsh(H)             # wartości własne w porządku rosnącym
lam_min, lam_max = wartosci_wlasne[0], wartosci_wlasne[-1]
lr_kryt = 2 / lam_max
print(f"lambda_min = {lam_min:.4f}, lambda_max = {lam_max:.4f}, uwarunkowanie = {lam_max / lam_min:.1f}")
print(f"teoretyczny próg rozbieżności lr_kryt = 2/lambda_max = {lr_kryt:.4f}")

# 2) trening dla każdego learning rate
EPOKI_LR = 40 if SZYBKI_TRYB else 200
lrs = [0.001, 0.01, 0.1, round(0.5 * lr_kryt, 4), round(0.98 * lr_kryt, 4), round(1.05 * lr_kryt, 4)]
historie = {}
for lr_ in lrs:
    _, _, hist = gd_linreg(X_tr_s, y_tr, lr=lr_, epoki=EPOKI_LR)
    historie[lr_] = hist

### Weryfikacja

In [ ]:
if historie:
    plt.figure(figsize=(9, 5))
    for lr_, h in historie.items():
        plt.plot(h, label=f"lr = {lr_}")
    plt.axhline(loss_opt, color="k", ls="--", lw=1, label="optimum")
    plt.yscale("log"); plt.ylim(loss_opt * 0.9, max(10, loss_opt * 100))
    plt.title(f"Wpływ learning rate na zbieżność GD (lr_kryt = {lr_kryt:.3f})")
    plt.xlabel("epoka"); plt.ylabel("MSE train (log)"); plt.legend(); plt.grid(alpha=0.3, which="both"); plt.show()

    print("lr        | MSE po ostatniej epoce | epoki do 1% od optimum")
    for lr_, h in historie.items():
        h = np.array(h)
        ok = np.where(h <= 1.01 * loss_opt)[0]
        print(f"{lr_:<9} | {h[-1]:>22.4g} | {ok[0] if len(ok) else 'nie osiągnięto'}")

# Zadanie 5.1 - Implementacja mini-batch SGD i porównanie rozmiarów batcha

In [ ]:
def minibatch_gd(X, y, lr=0.05, epoki=20, batch_size=32, seed=0):
    rng = np.random.default_rng(seed)
    n, d = X.shape
    w = np.zeros(d)
    b = 0.0
    historia, czasy = [], []
    t0 = time.perf_counter()
    for ep in range(epoki):
        perm = rng.permutation(n)                            # 1) nowa losowa permutacja co epokę
        for start in range(0, n, batch_size):                # 2) kolejne mini-batche
            idx = perm[start:start + batch_size]             #    (ostatni batch może być mniejszy)
            Xb_, yb_ = X[idx], y[idx]
            err = Xb_ @ w + b - yb_                          # 3) gradient MSE tylko na batchu
            dw = (2 / len(idx)) * (Xb_.T @ err)
            db = 2 * np.mean(err)
            w = w - lr * dw                                  #    krok SGD
            b = b - lr * db
        historia.append(np.mean((X @ w + b - y) ** 2))       # strata na całym zbiorze train
        czasy.append(time.perf_counter() - t0)
    return w, b, historia, czasy

### Weryfikacja

In [ ]:
EPOKI_MB = 4 if SZYBKI_TRYB else 30
wyniki_mb = {}
for bs in [16, 64, 256, n_tr]:
    _, _, h, c = minibatch_gd(X_tr_s, y_tr, lr=0.05, epoki=EPOKI_MB, batch_size=bs)
    nazwa = "full-batch" if bs == n_tr else f"B = {bs}"
    wyniki_mb[nazwa] = (h, c)
    print(f"{nazwa:>10}: kroków/epokę = {math.ceil(n_tr / bs):5d}, MSE końcowe = {h[-1]:.4f}, czas = {c[-1]:.2f} s")

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for nazwa, (h, c) in wyniki_mb.items():
    ax[0].plot(range(1, len(h) + 1), h, "o-", ms=3, label=nazwa)
    ax[1].plot(c, h, "o-", ms=3, label=nazwa)
for a_, xl in zip(ax, ["epoka", "czas [s]"]):
    a_.axhline(loss_opt, color="k", ls="--", lw=1, label="optimum")
    a_.set_yscale("log"); a_.set_xlabel(xl); a_.set_ylabel("MSE train (log)"); a_.grid(alpha=0.3, which="both"); a_.legend()
ax[0].set_title("Mini-batch SGD: strata vs epoka"); ax[1].set_title("Mini-batch SGD: strata vs czas")
plt.tight_layout(); plt.show()

# Zadanie 6.1 - Krzywa train vs walidacja w funkcji stopnia wielomianu

In [ ]:
stopnie = list(range(16))
mse_train_D, mse_val_D = [], []
for D in stopnie:
    wD = dopasuj_wielomian(x_trp, y_trp, D)                       # dopasowanie na zbiorze treningowym
    mse_train_D.append(mse(y_trp, poly_features(x_trp, D) @ wD))  # błąd na train
    mse_val_D.append(mse(y_vap, poly_features(x_vap, D) @ wD))    # błąd na walidacji
najlepszy_D = int(np.argmin(mse_val_D))      # stopnie zaczynają się od 0, więc indeks = stopień

### Weryfikacja

In [ ]:
if mse_val_D:
    print("najlepszy stopień wg walidacji:", najlepszy_D, " MSE val =", round(mse_val_D[najlepszy_D], 4))
    plt.figure(figsize=(8, 4.5))
    plt.plot(stopnie, mse_train_D, "o-", label="MSE train")
    plt.plot(stopnie, mse_val_D, "s-", label="MSE walidacja")
    plt.axvline(najlepszy_D, color="gray", ls=":", label=f"najlepszy D = {najlepszy_D}")
    plt.axhline(0.3 ** 2, color="k", ls="--", lw=1, label="wariancja szumu (0.09)")
    plt.yscale("log"); plt.title("Pojemność modelu: underfitting vs overfitting")
    plt.xlabel("stopień wielomianu D (hiperparametr)"); plt.ylabel("MSE (log)"); plt.legend(); plt.grid(alpha=0.3, which="both"); plt.show()

## Zadanie 6.2 - Regularyzacja L2 (ridge) jako "lekarstwo" na przeuczenie

In [ ]:
def dopasuj_ridge(x, y, stopien, lam):
    # (Phi^T Phi + n*lam*I) w = Phi^T y, bez kary na bias
    Phi = poly_features(x, stopien)
    n = len(x)
    kara = n * lam * np.eye(stopien + 1)
    kara[0, 0] = 0.0                                   # pierwsza kolumna (x^0) to bias - nie karzemy
    return np.linalg.solve(Phi.T @ Phi + kara, Phi.T @ y)   # solve zamiast jawnego odwracania macierzy

### Weryfikacja

In [ ]:
D_duzy = 15
lambdy = np.logspace(-12, 1, 27)
mse_tr_l, mse_va_l = [], []
for lam in lambdy:
    wl = dopasuj_ridge(x_trp, y_trp, D_duzy, lam)
    mse_tr_l.append(mse(y_trp, poly_features(x_trp, D_duzy) @ wl))
    mse_va_l.append(mse(y_vap, poly_features(x_vap, D_duzy) @ wl))
best_lam = lambdy[int(np.argmin(mse_va_l))]
print(f"najlepsza lambda = {best_lam:.2e}, MSE val = {min(mse_va_l):.4f}")

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].semilogx(lambdy, mse_tr_l, "o-", label="MSE train")
ax[0].semilogx(lambdy, mse_va_l, "s-", label="MSE walidacja")
ax[0].set_yscale("log"); ax[0].set_title(f"Ridge, D = {D_duzy}: błąd vs siła regularyzacji")
ax[0].set_xlabel("lambda (log)"); ax[0].set_ylabel("MSE (log)"); ax[0].legend(); ax[0].grid(alpha=0.3, which="both")
ax[1].plot(xx, prawdziwa_f(xx), "g--", label="prawdziwa funkcja")
ax[1].plot(xx, poly_features(xx, D_duzy) @ dopasuj_wielomian(x_trp, y_trp, D_duzy), "r", alpha=0.5, label="D=15 bez regularyzacji")
ax[1].plot(xx, poly_features(xx, D_duzy) @ dopasuj_ridge(x_trp, y_trp, D_duzy, best_lam), "b", lw=2, label=f"D=15, ridge lambda={best_lam:.1e}")
ax[1].scatter(x_trp, y_trp, edgecolors="k", facecolors="none", label="train")
ax[1].set_ylim(-2, 2); ax[1].set_title("Dopasowanie z regularyzacją L2"); ax[1].set_xlabel("x"); ax[1].set_ylabel("y"); ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()